# 01 — Explore HiRISE Imagery

This notebook walks through the raw HiRISE data, examines band statistics,
visualises the study area extents, and checks coregistration quality.

**Study sites:** Gasa, Palikir, Russell craters


In [ ]:
import sys
sys.path.insert(0, '..')  # add project root to path

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import rasterio
from pathlib import Path

from scripts.utils import load_config

cfg = load_config('../config.yaml')
print('Config loaded. Study areas:', list(cfg['study_area'].keys()))

In [ ]:
# ── List available HiRISE GeoTIFFs ──────────────────────────────────────────
hirise_dir = Path('../data/raw/hirise')
hirise_files = sorted(hirise_dir.glob('**/*.tif'))
print(f'Found {len(hirise_files)} HiRISE GeoTIFFs:')
for f in hirise_files[:10]:
    print(' ', f.relative_to(hirise_dir.parent.parent))

In [ ]:
# ── Load and visualise one HiRISE image ─────────────────────────────────────
if hirise_files:
    with rasterio.open(hirise_files[0]) as src:
        arr = src.read()          # (C, H, W)
        meta = src.meta
        bounds = src.bounds
        crs = src.crs
    
    print(f'Shape: {arr.shape}  |  dtype: {arr.dtype}')
    print(f'Bounds: {bounds}')
    print(f'CRS: {crs}')
    print(f'Band stats: min={arr.min():.1f}  max={arr.max():.1f}  mean={arr.mean():.1f}')
else:
    print('No HiRISE files found. Run: python main.py --step download')

In [ ]:
# ── False-colour RGB visualisation ──────────────────────────────────────────
def percentile_stretch(band, lo=2, hi=98):
    p_lo, p_hi = np.percentile(band, [lo, hi])
    return np.clip((band - p_lo) / (p_hi - p_lo + 1e-6), 0, 1)

if hirise_files and arr.shape[0] >= 3:
    rgb = np.stack([
        percentile_stretch(arr[0]),
        percentile_stretch(arr[1]),
        percentile_stretch(arr[2]),
    ], axis=-1)
    
    fig, axes = plt.subplots(1, 3, figsize=(16, 5))
    axes[0].imshow(rgb[:512, :512])
    axes[0].set_title('RGB (bands 1-2-3)')
    
    im = axes[1].imshow(percentile_stretch(arr[0, :512, :512]), cmap='gray')
    axes[1].set_title('Band 1 (RED)')
    plt.colorbar(im, ax=axes[1], fraction=0.046)
    
    # Histogram of band 1
    axes[2].hist(arr[0].ravel(), bins=100, color='steelblue', alpha=0.7)
    axes[2].set_title('Band 1 Histogram')
    axes[2].set_xlabel('DN value')
    
    plt.suptitle(f'HiRISE: {hirise_files[0].name}', fontsize=13)
    plt.tight_layout()
    plt.savefig('../data/outputs/reports/hirise_preview.png', dpi=120, bbox_inches='tight')
    plt.show()
    print('Preview saved.')

In [ ]:
# ── Band statistics table ────────────────────────────────────────────────────
if hirise_files:
    print(f'{'Band':<6} {'Min':>8} {'Max':>8} {'Mean':>10} {'Std':>10} {'p2':>8} {'p98':>8}')
    print('-' * 62)
    for i, band in enumerate(arr):
        p2, p98 = np.percentile(band, [2, 98])
        print(
            f'{i+1:<6} '
            f'{band.min():>8.1f} {band.max():>8.1f} '
            f'{band.mean():>10.2f} {band.std():>10.2f} '
            f'{p2:>8.1f} {p98:>8.1f}'
        )

In [ ]:
# ── Feature stack preview (if already generated) ─────────────────────────────
feat_dir = Path('../data/processed/feature_stacks')
feat_files = sorted(feat_dir.glob('*.tif'))
if feat_files:
    with rasterio.open(feat_files[0]) as src:
        feat = src.read().astype(np.float32)
    print(f'Feature stack shape: {feat.shape}  ({feat.shape[0]} channels)')
    
    fig, axes = plt.subplots(2, 4, figsize=(16, 8))
    channel_names = [
        'HiRISE R', 'HiRISE G', 'HiRISE B',
        'RSI', 'GLCM', 'CTX', 'Slope', 'Aspect'
    ]
    for i, ax in enumerate(axes.ravel()):
        if i < feat.shape[0]:
            band = feat[i, :256, :256]
            im = ax.imshow(percentile_stretch(band), cmap='viridis')
            ax.set_title(channel_names[i] if i < len(channel_names) else f'Ch {i}')
            ax.axis('off')
    plt.suptitle('Feature Stack Channels', fontsize=14)
    plt.tight_layout()
    plt.show()
else:
    print('No feature stacks found. Run: python main.py --step features')